# singleGBconvergence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/singleGB/singleGBconvergence.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
"""
Single GB propagation: frequency-sweep convergence in *relative energy*.

This script compares a single Gaussian-beam (GB) field to a k-Wave "ground truth"
solution for constant c(x)=1 on a periodic box. It measures the *sup over time* of
the energy-norm error and divides by the initial energy of the exact solution,
removing the spurious dimension dependence you were seeing when using L2.

Theory reference (unscaled energy norm; divide by initial energy to get a
dimension-free frequency rate ~ k^{-N/2}):
  Liu, Ralston, Yin (2019) "General Superpositions of Gaussian Beams and
  Propagation Errors", Math. Comp. 89 (2020), Theorems 1.2 and 3.1.

Practical notes:
- To reproduce the dimension-free rate, sweep ω (frequency) while keeping the
  initial beam width parameter α (Im M0) fixed O(1).
- If you instead tie α to ω, the beam's physical width changes in a way that
  contaminates the normalization and reintroduces an explicit d-dependence.

Author: (rewritten for energy-norm normalization)
"""

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from functools import partial
from time import time
from pathlib import Path

from beamax import geometry, plotter, utils
from beamax.gb import core, gb_utils, gb_solvers
from beamax.solvers.kwave_solver import TimedKWaveSolver

from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions

# ------------------------------------------------------------
# JAX + paths
# ------------------------------------------------------------
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.config.update(
    "jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir"
)

ROOT_DIR = utils.detect_root()
CACHE_DIR = Path(ROOT_DIR / "cache")
PLOT_DIR = Path(ROOT_DIR / "plots")
CACHE_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

pltgb = plotter.PlotHelper()

In [ ]:
# ------------------------------------------------------------
# Domain / physics
# ------------------------------------------------------------
b = 2  # number of beams in the sum (single-beam study => b = 1)
d = 1  # spatial dimension (set 1 or 2; code is d-agnostic)
N = (1024,) * d
dx = (1.0 / N[0],) * d
periodic = (True,) * d


def c(x):
    # constant speed (choose 1 so k-Wave and GB agree cleanly on propagation)
    return 1.0 + 0.0 * x[..., 0]


cfl = 0.3
lam = 0.0  # unused GB parameter here (left to keep interfaces intact)
domain = geometry.Domain(N=N, dx=dx, c=c, cfl=cfl, periodic=periodic)
XY = domain.grid
ts = domain.generate_time_domain()
dt = ts[1] - ts[0]

# Sensors: everywhere (so we can compare fields pointwise)
sensors_all = jnp.ones(N)
sensors = geometry.Sensor(domain=domain, binary_mask=sensors_all)

# k-Wave as "exact" propagation
simulation_options = SimulationOptions(
    data_cast="double",
    smooth_p0=False,
    save_to_disk=True,
)
execution_options = SimulationExecutionOptions(
    is_gpu_simulation=False, delete_data=False, verbose_level=0
)
kwave_solver = TimedKWaveSolver(simulation_options, execution_options)

# Initial GB parameters (ray, direction, amplitude)
mode = jnp.array([1, -1])
x0 = (domain.grid_size * 0.5).repeat(b).reshape(b, d)  # center of box
p0 = jnp.zeros((b, d))
p0 = p0.at[:, 0].set(1.0)
p0 = p0 / jnp.linalg.norm(p0, axis=-1, keepdims=True)  # unit direction
a0 = jnp.ones((b,))  # O(1) amplitude

solver = gb_solvers.solve_hom_general
solver_config = None

# Print a correct spatial Nyquist in cycles per unit length (for sanity checks)
nyquist_cycles = 1.0 / (2.0 * dx[0])
print(f"Spatial Nyquist (cycles/unit length): {nyquist_cycles:.3f}")


# ------------------------------------------------------------
# Energy norm utilities (unscaled)
#   E[v]^2 = (1/2) ∫ (|v_t|^2 + |∇v|^2) dx
# ------------------------------------------------------------
def _central_time_derivative(u, dt):
    # u: (T, n1, n2, ..., nd)
    ut_mid = (u[2:] - u[:-2]) / (2.0 * dt)
    ut0 = (u[1] - u[0]) / dt
    utT = (u[-1] - u[-2]) / dt
    return jnp.concatenate([ut0[None, ...], ut_mid, utT[None, ...]], axis=0)


def _periodic_gradients(u, dx):
    # u: (T, n1, n2, ..., nd) -> list of d arrays of same shape as u
    grads = []
    for ax, h in enumerate(dx, start=1):  # spatial axes start at 1 (axis 0 is time)
        grads.append((jnp.roll(u, -1, axis=ax) - jnp.roll(u, 1, axis=ax)) / (2.0 * h))
    return grads


def energy_over_time(u, dt, dx):
    """Unscaled energy norm E[u(t)] for each time t on a periodic grid."""
    ut = _central_time_derivative(u, dt)
    grads = _periodic_gradients(u, dx)
    vol = jnp.prod(jnp.array(dx))
    integrand = ut**2
    for g in grads:
        integrand = integrand + g**2
    # integrate over space, sqrt, and scale by 1/2
    axis = tuple(range(1, u.ndim))
    Et = jnp.sqrt(0.5 * jnp.sum(integrand, axis=axis) * vol)
    return Et  # shape (T,)


def initial_energy_from_u0(u0, dx):
    """Unscaled initial energy if initial velocity is zero: E0 = sqrt( 1/2 ∫ |∇u0|^2 dx )."""
    # Add a fake time axis to reuse periodic_gradients
    grads0 = _periodic_gradients(u0[None, ...], dx)
    vol = jnp.prod(jnp.array(dx))
    gsum0 = jnp.sum(jnp.stack([g[0] ** 2 for g in grads0], axis=0), axis=0)
    E0 = jnp.sqrt(0.5 * jnp.sum(gsum0) * vol)
    return E0


# ------------------------------------------------------------
# GB field assembly
# ------------------------------------------------------------
@partial(jax.jit, static_argnums=(6, 11, 12))
def compute_gb_sum(
    x0, p0, M0, a0, ω0, mode, c, ts, XY, domain_size, periodic, solver, solver_config
):
    # Sum over beams and return the real part
    return jnp.sum(
        core.compute_gaussian_beam(
            x0,
            p0,
            M0,
            a0,
            ω0,
            mode,
            c,
            lam,
            ts,
            XY,
            domain_size,
            jnp.array(periodic),
            solver,
            solver_config,
        ),
        axis=-1,
    ).real  # shape: (T, n1, ..., nd)

In [ ]:
# ------------------------------------------------------------
# Frequency sweep (ω only) with fixed α
# ------------------------------------------------------------
omega_vals = jnp.logspace(0, jnp.log10(float(N[0])), 10)

# omega_vals = [jnp.logspace(1.0, 2.9, 10)]  # feel free to refine

# Optional alternative sweeps (not used in the main plot/regression):
# alpha_vals = jnp.logspace(1.0, 2.9, 10)
# both_vals  = jnp.logspace(1.0, 2.9, 10)

rel_energy_errors = []  # sup_t E(error)/E0 per ω
gb_runtimes = []
kwave_runtimes = []

for idx, omega in enumerate(omega_vals):
    ω0 = jnp.ones((b,)) * omega
    # Keep α = Im(M0) fixed O(1): Im(M0) = α0 I with α0 = 1
    alpha0 = 100j * jnp.ones((b, d))

    # Build M0 and solve GB ray/transport system (for completeness; xt,pt,mt,at unused here)
    M0 = gb_utils.prepare_M0(alpha0, None)
    _ = gb_utils.is_diagonal(M0)
    (xt, pt, mt, at) = solver(x0, p0, M0, a0, mode, ts, c, lam, solver_config)

    # GB field on whole time grid
    t1 = time()
    u_gb_all = compute_gb_sum(
        x0,
        p0,
        M0,
        a0,
        ω0,
        mode,
        c,
        ts,
        XY,
        domain.grid_size,
        periodic,
        solver,
        solver_config,
    )
    t2 = time()
    gb_runtimes.append(t2 - t1)
    print(f"[ω={float(omega):.3f}] GB runtime: {t2 - t1:.3f}s")

    # Initial field (pressure) for k-Wave
    u0_init = u_gb_all[0, ...]

    # Handle 1D case by embedding into 2D for k-Wave if needed (as in your original)
    if d == 1:
        N_kw = (N[0], 1)
        dx_kw = (dx[0],) * 2
        periodic_kw = (periodic[0],) * 2
        domain_kw = geometry.Domain(
            N=N_kw, dx=dx_kw, c=c, cfl=cfl, periodic=periodic_kw
        )
        sensors_all_kw = jnp.ones(N_kw)
        u0_init_kw = u0_init.reshape(N_kw)
    else:
        domain_kw = domain
        sensors_all_kw = sensors_all
        u0_init_kw = u0_init

    # Propagate with k-Wave
    pt_kwave_all, runtime = kwave_solver.forward(
        u0_init_kw, domain_kw, sensors_all_kw, ts
    )
    kwave_runtimes.append(runtime)

    # Reshape back to (T, n1, ..., nd)
    pt_kwave_all = jnp.transpose(
        pt_kwave_all.reshape(-1, *N), (0, len(N), *(range(1, len(N))))
    )

    # if d == 1:
    #     plt.plot(u0_init)
    #     plt.show()
    # elif d == 2:
    #     plt.imshow(u0_init)
    #     plt.show()
    # #assert False

    # Sanity check: initial states should match
    assert jnp.allclose(
        pt_kwave_all[0, ...], u0_init
    ), "k-Wave initial condition mismatch."

    # -------------------------------
    # Energy errors and normalization
    # -------------------------------
    diff_all = pt_kwave_all - u_gb_all
    E_err_t = energy_over_time(diff_all, dt, dx)  # E(error)(t)
    E0 = initial_energy_from_u0(u_gb_all[0, ...], dx)  # equals exact initial energy

    rel_E = jnp.max(E_err_t) / E0
    rel_energy_errors.append(float(rel_E))

# Convert to arrays for plotting/regression
omega_vals = jnp.array(omega_vals)
rel_energy_errors = jnp.array(rel_energy_errors)

In [ ]:
omega_vals = omega_vals[:-2]
rel_energy_errors = rel_energy_errors[:-2]
# ------------------------------------------------------------
# Plot: relative energy error vs frequency (dimension-free)
# ------------------------------------------------------------
plt.figure(figsize=(8, 6))
freq_range = jnp.logspace(
    jnp.log10(float(omega_vals.min())), jnp.log10(float(omega_vals.max())), 200
)

# Reference ~ K^{-1/2}
Cref = 2.0 * float(rel_energy_errors.max())
plt.loglog(
    freq_range,
    Cref * (freq_range ** (-0.5)),
    alpha=0.5,
    label=r"reference $\propto \omega^{-1/2}$",
)

plt.loglog(
    omega_vals,
    rel_energy_errors,
    marker="o",
    linestyle="-",
    label=r"$\omega$",
)
plt.xlabel(r"$\omega$")
plt.ylabel(r"$\max_t\, \|u - u_{\rm GB}\|_{E} \,/\, \|u(\cdot,0)\|_{E}$")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / f"relative_energy_vs_frequency_d{d}.png", dpi=200)
plt.show()

# ------------------------------------------------------------
# Fit slope on log-log (should be ≈ -1/2 for first-order beams)
# ------------------------------------------------------------
p = jnp.polyfit(jnp.log(omega_vals), jnp.log(rel_energy_errors), 1)
print(
    f"Measured slope (relative energy vs K): {float(p[0]):.4f}  (expected ≈ -0.5 for N=1)"
)
print(
    f"Mean GB runtime per ω: {sum(gb_runtimes) / len(gb_runtimes):.3f}s ; "
    f"Mean k-Wave runtime per ω: {sum(kwave_runtimes) / len(kwave_runtimes):.3f}s"
)